# Image Model Deployment to Databricks Model Serving

This notebook deploys a registered image model to Databricks Model Serving on GPU_LARGE instances.

## Install Required Packages

In [0]:
%pip install mlflow[databricks] databricks-sdk --upgrade
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 67.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.7/774.7 kB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 164.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 137.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 609.9/609.9 kB 76.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 137.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.5/803.5 kB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 198.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.

## Setup Widgets for Configuration

In [0]:
dbutils.widgets.text("catalog", "main", "Catalog Name")
dbutils.widgets.text("schema", "default", "Schema Name")
dbutils.widgets.text("model_name", "image_model", "Model Name")
dbutils.widgets.text("endpoint_name", "image_model_endpoint", "Endpoint Name")
dbutils.widgets.text("model_alias", "staging", "Model stage")

## Read Configuration from Widgets

In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
model_name = dbutils.widgets.get("model_name")
endpoint_name = dbutils.widgets.get("endpoint_name")
model_stage = dbutils.widgets.get("model_alias")

registered_model_name = f"{catalog}.{schema}.{model_name}"

print(f"Catalog: {catalog}")
print(f"Schema: {schema}")
print(f"Registered Model: {registered_model_name}")
print(f"Model Stage: {model_stage}")
print(f"Endpoint Name: {endpoint_name}")

Catalog: uc_sriharsha_jana
Schema: default
Registered Model: uc_sriharsha_jana.default.qwen_image_edit_model
Model Stage: staging
Endpoint Name: qwen_image_edit_model


## Initialize Databricks Workspace Client

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    AutoCaptureConfigInput,
    ServingModelWorkloadType
)

w = WorkspaceClient()
print(f"Workspace URL: {w.config.host}")

model_version = w.model_versions.get_by_alias(registered_model_name, model_stage).version
model_version

Workspace URL: https://adb-984752964297111.11.azuredatabricks.net


2

## Deploy Model to Serving Endpoint

This will create or update a model serving endpoint with GPU_LARGE workload size.

In [0]:
# Check if endpoint exists
try:
    existing_endpoint = w.serving_endpoints.get(endpoint_name)
    print(f"Endpoint '{endpoint_name}' already exists. Updating...")
    endpoint_exists = True
except Exception as e:
    print(f"Endpoint '{endpoint_name}' does not exist. Creating new endpoint...")
    endpoint_exists = False

Endpoint 'qwen_image_edit_model' already exists. Updating...


In [0]:
# Configure the served entity
served_entity = ServedEntityInput(
    entity_name=registered_model_name,
    entity_version=model_version,
    workload_size="Small",
    workload_type=ServingModelWorkloadType.GPU_LARGE,
    scale_to_zero_enabled=True,
)

if endpoint_exists:
    # Update existing endpoint
    w.serving_endpoints.update_config_and_wait(
        name=endpoint_name,
        served_entities=[served_entity]
    )
    print(f"Endpoint '{endpoint_name}' updated successfully!")
else:
    # Create new endpoint
    w.serving_endpoints.create_and_wait(
        name=endpoint_name,
        config=EndpointCoreConfigInput(
            name=endpoint_name,
            served_entities=[served_entity]
        )
    )
    print(f"Endpoint '{endpoint_name}' created successfully!")

---------------------------------------------------------------------------
TimeoutError                              Traceback (most recent call last)
File <command-8092587260746133>, line 12
      2 served_entity = ServedEntityInput(
      3     entity_name=registered_model_name,
      4     entity_version=model_version,
   (...)
      7     scale_to_zero_enabled=True,
      8 )
     10 if endpoint_exists:
     11     # Update existing endpoint
---> 12     w.serving_endpoints.update_config_and_wait(
     13         name=endpoint_name,
     14         served_entities=[served_entity]
     15     )
     16     print(f"Endpoint '{endpoint_name}' updated successfully!")
     17 else:
     18     # Create new endpoint

File /local_disk0/.ephemeral_nfs/envs/pythonEnv-83024c34-da01-4054-aba5-5ea86dca2011/lib/python3.12/site-packages/databricks/sdk/service/serving.py:4732, in ServingEndpointsAPI.update_config_and_wait(self, name, auto_capture_config, served_entities, served_models, traffic_co

## Get Endpoint Details

In [0]:
endpoint = w.serving_endpoints.get(endpoint_name)
print(f"\nEndpoint Name: {endpoint.name}")
print(f"Endpoint State: {endpoint.state.ready}")
print(f"Endpoint URL: {w.config.host}/serving-endpoints/{endpoint_name}/invocations")